# 第 4 周练习：Python → C++ 优化器 + 复杂度注释

## 练习目标

把一段 Python 函数交给大模型，让它输出：

1. **优化后的 C++17 代码**（放在 ` ```cpp ` 代码块里）
2. **复杂度分析**（时间 + 空间）
3. **微优化说明**（若干要点）
4. **输入 / 输出示例**

对应第 4 周常见主题：用 system / user prompt 约束输出格式，并用流式（streaming）在笔记本里边生成边展示。

## 怎么跑

1. 准备 `.env`，写入 `OPENAI_API_KEY`（本笔记本经 OpenRouter 的 `base_url` 调用）
2. 从上到下运行单元格；可在「Python 函数输入」格改 `python_code`
3. 默认跑流式版 `convert_and_optimize_stream`；也可取消注释非流式那一格做对比


In [30]:
# ========== 导入：环境变量、笔记本展示、OpenAI 客户端 ==========

# 导入标准库 os：用 getenv 读 OPENAI_API_KEY
import os
# 从 dotenv 导入 load_dotenv：把 .env 读进进程环境，避免密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入 Markdown / display / update_display：流式刷新笔记本输出
from IPython.display import Markdown, display, update_display
# 从 openai 导入 OpenAI：Chat Completions 客户端（可指向 OpenRouter）
from openai import OpenAI


In [31]:
# ========== 加载密钥并创建客户端（经 OpenRouter 兼容端点） ==========

# override=True：.env 里的值覆盖已有同名环境变量
load_dotenv(override=True)
# 从环境读取 API Key 字符串
api_key = os.getenv('OPENAI_API_KEY')

# 三种自检：缺失 / 首尾空白 / 看起来正常（提示文案保持英文原样）
if not api_key:
    print('No API key found. Please add OPENAI_API_KEY to your .env file.')
elif api_key.strip() != api_key:
    print('API key has leading/trailing whitespace. Please remove it.')
else:
    print('API key looks good!')

# 创建客户端：base_url 指向 OpenRouter 的 OpenAI 兼容 API
openai = OpenAI(
    api_key=api_key,
    base_url='https://openrouter.ai/api/v1',
)


API key looks good!


In [32]:
# ========== 选型：本练习固定使用的模型 id ==========

# 模型名必须与 OpenRouter / 上游可用模型一致；改这里即可全局切换
MODEL = 'gpt-4.1-mini'


In [33]:
# ========== 待转换的 Python 示例：可换成你自己的函数 ==========

# 三引号字符串保存整段源码，后面作为 user 消息内容发给模型
python_code = '''
def top_k_frequent(nums, k):
    counts = {}
    for n in nums:
        counts[n] = counts.get(n, 0) + 1
    return sorted(counts.keys(), key=lambda x: counts[x], reverse=True)[:k]
'''


In [34]:
# ========== Prompt 模板：system 定格式，user 前缀引导「转换这段 Python」 ==========

# SYSTEM_PROMPT：角色 + 必须出现的 Markdown 标题 + 代码块规则（字符串勿改，影响行为）
SYSTEM_PROMPT = '''
You are a senior C++ engineer and performance tuner.
Convert Python code to optimized C++17.
You MUST return markdown with these exact headings (use ##):
# C++代码
# 复杂
# 优化笔记
# 输入/输出示例
Rules:
- C++ code must be inside a single fenced code block with ```cpp.
- Complexity must include time and space.
- Optimization Notes must be 2-4 bullet points.
- Example I/O must show one sample input and output.
If you fail to follow the format exactly, the answer is invalid.
'''

# USER_PROMPT_PREFIX：拼在具体 Python 源码前面的固定引导语
USER_PROMPT_PREFIX = '''
Convert this Python function to optimized C++ and analyze it:

'''


In [35]:
# ========== messages 组装：OpenAI 风格的 system + user 两条消息 ==========

def messages_for(code_text: str):
    # 返回 list[dict]，供 chat.completions.create(messages=...) 直接使用
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': USER_PROMPT_PREFIX + code_text},
    ]


In [36]:
# ========== 后处理：尽量补齐必需标题，并在缺 cpp 围栏时粗暴包一层 ==========

def enforce_format(md: str) -> str:
    # 期望模型输出的四个二级标题（与 SYSTEM_PROMPT 约束对齐；缺则前置补标题）
    required = ['## C++ Code', '## Complexity', '## Optimization Notes', '## Example I/O']
    for h in required:
        if h not in md:
            md = h + '\n' + md
    # 若全文没有 ```cpp 围栏：尝试在「## C++ Code」后插入 naive wrap
    if '```cpp' not in md:
        # naive wrap：按第一次出现的标题切开，后半包进 cpp 代码块
        parts = md.split('## C++ Code', 1)
        if len(parts) == 2:
            before, after = parts
            md = before + '## C++ Code\n```cpp\n' + after.strip() + '\n```'
    return md


In [37]:
# ========== 非流式：一次拿齐回复，再走 enforce_format ==========

def convert_and_optimize(code_text: str) -> str:
    # 调用 Chat Completions；max_tokens=900 限制单次生成长度
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages_for(code_text),
        max_tokens=900,
    )
    # 取助手文本并做格式兜底，返回 Markdown 字符串
    return enforce_format(response.choices[0].message.content)


In [38]:
# ========== 流式：边收 token 边刷新 Markdown，最后再 enforce_format ==========

def convert_and_optimize_stream(code_text: str):
    # stream=True：按 chunk 推送；max_tokens 与非流式一致
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=messages_for(code_text),
        stream=True,
        max_tokens=900,
    )
    # 累积完整回复
    response = ''
    # 先占位一个可更新的 Markdown 显示句柄（display_id=True）
    display_handle = display(Markdown(''), display_id=True)
    for chunk in stream:
        # delta.content 可能为 None（某些 chunk 只有 role/finish），用 or '' 兜底
        response += chunk.choices[0].delta.content or ''
        # 每来一块就刷新同一 display_id，实现「打字机」效果
        update_display(Markdown(response), display_id=display_handle.display_id)

    # 流结束后用 enforce_format 再渲染终稿
    final_md = enforce_format(response)
    update_display(Markdown(final_md), display_id=display_handle.display_id)

# 对上面定义的 python_code 跑一遍流式转换
convert_and_optimize_stream(python_code)


## C++ Code
```cpp
#include <vector>
#include <unordered_map>
#include <algorithm>

std::vector<int> top_k_frequent(const std::vector<int>& nums, int k) {
    std::unordered_map<int, int> counts;
    counts.reserve(nums.size());  // Reserve to reduce rehashing

    for (int n : nums) {
        ++counts[n];
    }

    std::vector<int> unique;
    unique.reserve(counts.size());
    for (const auto& kvp : counts) {
        unique.push_back(kvp.first);
    }

    std::nth_element(unique.begin(), unique.begin() + k, unique.end(),
        [&counts](int a, int b) { return counts[a] > counts[b]; });

    unique.resize(k);
    std::sort(unique.begin(), unique.end(),
        [&counts](int a, int b) { return counts[a] > counts[b]; });

    return unique;
}
```

## Complexity
- Time: O(N + M log k), where N is the size of `nums`, M is the number of unique elements (M ≤ N). Counting frequencies is O(N), `nth_element` is average O(M), resizing and sorting top k elements is O(k log k).
- Space: O(M) for storing frequency counts and the unique elements vector.

## Optimization Notes
- Used `unordered_map` for O(1) average frequency counting.
- Reserved capacity for the map and vector to minimize reallocations.
- Used `nth_element` to partially sort only the top k, improving performance over full sorting.
- Sorted only the top k elements to finalize order, thus reducing overhead.

## Example I/O
Input: nums = {1,1,1,2,2,3}, k = 2  
Output: {1, 2}

In [20]:
# 非流式备选：取消下一行注释即可；默认注释掉以免重复打 API
# 显示（Markdown（convert_and_optimize（python_code）））


## C++ Code
```cpp
#include <vector>
#include <unordered_map>
#include <queue>
#include <algorithm>

std::vector<int> top_k_frequent(const std::vector<int>& nums, int k) {
    // Count frequencies
    std::unordered_map<int, int> counts;
    for (int n : nums) {
        ++counts[n];
    }

    // Use a min-heap to keep track of top k frequent elements
    // The heap will store pairs of (frequency, element)
    auto cmp = [](const std::pair<int,int>& a, const std::pair<int,int>& b) {
        return a.first > b.first; // min-heap based on frequency
    };
    std::priority_queue<std::pair<int,int>, std::vector<std::pair<int,int>>, decltype(cmp)> min_heap(cmp);

    for (const auto& kv : counts) {
        min_heap.emplace(kv.second, kv.first);
        if ((int)min_heap.size() > k) {
            min_heap.pop();
        }
    }

    // Extract results from the heap (in ascending frequency order)
    std::vector<int> result;
    result.reserve(k);
    while (!min_heap.empty()) {
        result.push_back(min_heap.top().second);
        min_heap.pop();
    }
    // Reverse to get descending frequency order
    std::reverse(result.begin(), result.end());
    return result;
}
```

## Complexity
- Time complexity: O(N log k), where N is the number of elements in nums. Counting takes O(N), and maintaining the heap of size k takes O(log k) per insert.
- Space complexity: O(N) for the frequency hash map and O(k) for the heap.

## Optimization Notes
- Using a min-heap of size k reduces sorting cost compared to sorting all unique elements.
- The hash map provides O(1) average time complexity for frequency counts.
- Reserving result vector capacity avoids repeated reallocations.
- The approach balances speed and memory for large inputs with many unique elements.

## Example I/O
Input:  
`nums = {1,1,1,2,2,3}, k = 2`

Output:  
`{1, 2}`  
Explanation: 1 appears 3 times, 2 appears 2 times, 3 appears once. Top 2 frequent are 1 and 2.